In [1]:
import os
import json
import pandas as pd
import pyodbc as ob
import numpy as np

In [2]:
server = "SERPSQL"
database = "UKSERPUKLLC"
cnxn_str ="DRIVER={}; SERVER={}; DATABASE={}; Trusted_Connection=yes".format(
            "{ODBC Driver 17 for SQL Server}",
            server,
            database
        )
con = ob.connect(cnxn_str)

In [3]:
def read_table(llc_proj, table, con):
    
    return pd.read_sql(f''' SELECT * FROM {llc_proj}.{table}''', con=con)

In [4]:
data_dir_ = 'S:\LLC_0028\data\harmonised_covariates'
llc_proj='LLC_0028'
table = 'RETURNED_sociodemo_harmonised_selfreport_v0006_20230721'

In [5]:
dta = read_table( llc_proj, table, con)

In [10]:
dta_p = dta.pivot(columns = 'object',values = 'value', index='llc_0028_stud_id')

In [11]:
dta_p = dta_p.reset_index()

In [14]:
dta_p = dta_p.rename(columns = {'llc_0028_stud_id':'LLC_0028_stud_id'})

#### Age is missing for lots of BiB participants 

####  BIB

In [16]:
table = 'BIB_SOCIODEMO_v0006_20221025'
bib = read_table( llc_proj, table, con)

In [17]:
bib.partial_dob = pd.to_datetime(bib.partial_dob, format='%Y-%m')

In [18]:
symptom_date = pd.to_datetime('2020-09-01', format='%Y-%m')
bib['llc_age'] = [(symptom_date-dob).days/365 for dob in bib.partial_dob.values]

In [19]:
dta_p = dta_p.merge(bib[['LLC_0028_stud_id','llc_age']],
                    left_on = ['LLC_0028_stud_id'],
                    right_on = ['LLC_0028_stud_id'],
                    how = 'left')

In [24]:
llc_age = []
for i,v in enumerate(dta_p.llc_age_x.values):
    if np.isnan(v):
        bib_age = dta_p.llc_age_y.values[i]
        if not np.isnan(bib_age):
            llc_age.append(bib_age)
        else:
            llc_age.append(v)
    else:
        llc_age.append(v)
dta_p['llc_age'] = llc_age
dta_p = dta_p.drop(['llc_age_x','llc_age_y'],axis=1)

## Save 

In [26]:
dta_p.to_csv('S:/LLC_0028/data/harmonised_covariates/llc_0028_harmonised_covariates_returned.csv')